In [1]:
# Les importations
from datasets import load_dataset,Dataset, DatasetDict, concatenate_datasets


/home/koubra/Documents/Projet_IA/Chadian arabic update/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Chargement du dataset

In [2]:
def dataset_loader(path) :
    with open(path, "r") as file:
        data = file.readlines()

    data_clean = [line for line in data if line.strip()]

    return data_clean


In [3]:
PATH = "../Data/full_dataset/"
data_arb_bible = dataset_loader(PATH + "arabe_tchad3.txt")
data_fr_bible = dataset_loader(PATH + "V_francais_3.txt")

## Ceux non biblique
data_arb_not_bible = dataset_loader(PATH + "arabic_not_bible")
data_fr_not_bible = dataset_loader(PATH + "french_not_bible")

In [4]:
# Création du dictionnaire
dataset_dict_bible = {"translation": [{"arb": a, "fr": f} for a, f in zip(data_arb_bible, data_fr_bible)]}
dataset_dict_not_bible = {"translation": [{"arb": a, "fr": f} for a, f in zip(data_arb_not_bible, data_fr_not_bible)]}


### Diviosion du dataset en train, test et validation

In [5]:
def equal_split(data_dict1: dict, data_dict2: dict | None = None) -> DatasetDict:
    def split_dataset(data_dict: dict) -> DatasetDict:
        dataset = Dataset.from_dict(data_dict)
        train_test = dataset.train_test_split(
            test_size=0.2,
            seed=42,
            shuffle=True,
        )
        val_test = train_test["test"].train_test_split(
            test_size=0.5,
            seed=42,
            shuffle=True,
        )

        return DatasetDict({
            "train": train_test["train"],
            "val": val_test["train"],
            "test": val_test["test"],
        })

    datasets = [split_dataset(data_dict1)]

    if data_dict2 is not None:
        datasets.append(split_dataset(data_dict2))

    return DatasetDict({
        split: concatenate_datasets([dataset[split] for dataset in datasets])
        if len(datasets) > 1
        else datasets[0][split]
        for split in ("train", "val", "test")
    })


In [6]:
train_test_val_dataset = equal_split(dataset_dict_bible, dataset_dict_not_bible)
train_test_val_dataset

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 30950
    })
    val: Dataset({
        features: ['translation'],
        num_rows: 3869
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 3869
    })
})

In [7]:
train_test_val_dataset.save_to_disk("../Data/train_test_split")

Saving the dataset (0/1 shards):   0%|          | 0/30950 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 3869/3869 [00:00<00:00, 65223.08 examples/s]
